# 02 — Character-Level Text Generation with LSTM
### Deep Learning Week 3: Autoregressive Language Modeling & Temperature Sampling

---

## 1. Conceptual Introduction: Autoregressive Language Modeling

Text generation is formulated as a sequential prediction task. Given an initial sequence of characters:
$$x_1, x_2, \dots, x_t$$
the language model predicts the categorical probability distribution over the next character:
$$P(x_{t+1} = c \mid x_1, x_2, \dots, x_t) \quad \text{for each character } c \in \mathcal{V}$$

### Character-Level vs. Word-Level Modeling
| Feature | Character-Level Language Model | Word-Level Language Model |
| :--- | :--- | :--- |
| **Vocabulary Size ($|\mathcal{V}|$)** | Tiny (~65 to 100 characters) | Massive (10,000 to 100,000+ words) |
| **Out-of-Vocabulary (OOV)** | 0% (every text decomposes into characters) | Significant (unknown words require `<UNK>`) |
| **Sequence Length ($T$)** | Long (a single sentence is 50–200 characters) | Short (a sentence is 10–30 words) |
| **Task Complexity** | Must learn spelling, words, punctuation, and grammar | Learns syntax and semantics directly |

### Temperature-Based Sampling
When generating the next character, taking the greedy argmax $\arg\max_c P(x_{t+1}=c)$ causes repetitive, degenerate loops. Sampling randomly from the raw softmax probabilities can produce incoherent gibberish.

**Temperature ($T > 0$)** rescales the unnormalized model logits $z_i$ or log-probabilities:
$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)} = \frac{\exp(\log(p_i) / T)}{\sum_j \exp(\log(p_j) / T)}$$

- **$T \to 0$ (Low Temperature):** Probability mass concentrates on the highest logit. The model becomes conservative, confident, and grammatically strict, but repetitive.
- **$T = 1.0$ (Neutral Temperature):** Standard softmax distribution as learned during training.
- **$T > 1.2$ (High Temperature):** Flattens the distribution toward uniform random selection. Output is diverse and creative, but prone to invented words and syntactic breakdown.

In this notebook, we train an LSTM on William Shakespeare's public-domain plays and systematically evaluate temperature sampling.


In [1]:
# 2. Imports and Environment Setup
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

# Add Week-3 to sys.path to access reusable model_utils
current_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from model_utils import (
    build_char_lstm_model,
    build_vocabulary,
    sample_next_char,
    generate_text,
    load_artifacts,
)

# Set random seed
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")


TensorFlow Version: 2.21.0


## 3. Corpus Selection & Preprocessing: TinyShakespeare

We use the public-domain Shakespeare corpus (`shakespeare.txt`), derived from Project Gutenberg.
The text contains plays with character dialogue, act markers, and verse structure.


In [2]:
# 3.1 Load and Inspect Shakespeare Corpus
data_path = os.path.join(current_dir, "data", "shakespeare.txt")
if not os.path.exists(data_path):
    # Fallback path if running inside Week-3 subfolder
    data_path = os.path.join("..", "data", "shakespeare.txt")

with open(data_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Total corpus length: {len(raw_text):,} characters")
print("\n--- Corpus Excerpt (First 250 characters) ---")
print(raw_text[:250])
print("---------------------------------------------")


Total corpus length: 1,115,394 characters

--- Corpus Excerpt (First 250 characters) ---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

---------------------------------------------


### 3.2 Vocabulary Construction and Bidirectional Mapping

We extract all unique characters and construct:
- `char_to_idx`: maps characters to integer token indices $[0, |\mathcal{V}| - 1]$.
- `idx_to_char`: maps integer indices back to character strings.


In [3]:
# 3.2 Build Character Vocabulary
# Use first 120,000 characters for fast CPU training and consistent demonstration
training_text = raw_text[:120000]
unique_chars, char_to_idx, idx_to_char = build_vocabulary(training_text)
vocab_size = len(unique_chars)

print(f"Training slice length: {len(training_text):,} characters")
print(f"Unique characters ({vocab_size}):")
print("".join(unique_chars))


Training slice length: 120,000 characters
Unique characters (61):

 !&',-.:;?ABCDEFGHIJKLMNOPQRSTUVWYabcdefghijklmnopqrstuvwxyz


### 3.3 Sequence Generation (Sliding Window)

We cut the text into overlapping chunks of length `SEQ_LENGTH = 40` with stride `STEP = 3`.


In [4]:
# 3.3 Create Fixed-Length Sliding Windows
SEQ_LENGTH = 40
STEP = 3

sentences = []
next_chars = []
for i in range(0, len(training_text) - SEQ_LENGTH, STEP):
    sentences.append(training_text[i : i + SEQ_LENGTH])
    next_chars.append(training_text[i + SEQ_LENGTH])

print(f"Total training sequences: {len(sentences):,}")
print(f"Example input sentence: '{sentences[0]}'")
print(f"Example target next char: '{next_chars[0]}'")

# Vectorize into numpy integer arrays
X = np.zeros((len(sentences), SEQ_LENGTH), dtype=np.int32)
y = np.zeros((len(sentences),), dtype=np.int32)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t] = char_to_idx[char]
    y[i] = char_to_idx[next_chars[i]]

print(f"X shape: {X.shape}, y shape: {y.shape}")


Total training sequences: 39,987
Example input sentence: 'First Citizen:
Before we proceed any fur'
Example target next char: 't'


X shape: (39987, 40), y shape: (39987,)


## 4. LSTM Model Architecture

The neural network consists of:
1. `Input`: Shape `(40,)` containing character indices.
2. `Embedding(vocab_size, 64)`: Maps each character to a 64-dimensional vector.
3. `LSTM(128, return_sequences=False)`: Summarizes the 40-character sequence into a 128-dimensional hidden state vector.
4. `Dropout(0.2)`: Regularization to prevent verbatim sequence memorization.
5. `Dense(vocab_size, activation='softmax')`: Predicts the probability distribution over all unique characters.


In [5]:
# 4.1 Build or Load the Pre-Trained Model
models_dir = os.path.join(current_dir, "models")
if not os.path.exists(models_dir):
    models_dir = os.path.join(current_dir, "Week-3", "models")
model_file = os.path.join(models_dir, "text_generator.keras")

if os.path.exists(model_file):
    print("Loading existing trained model and artifacts from models/...")
    model, char_to_idx, idx_to_char, config = load_artifacts(models_dir)
else:
    print("Building fresh character LSTM model...")
    model = build_char_lstm_model(
        vocab_size=vocab_size,
        seq_length=SEQ_LENGTH,
        embedding_dim=64,
        lstm_units=128,
        dropout_rate=0.2,
        learning_rate=0.005,
    )

model.summary()


Loading existing trained model and artifacts from models/...


Model: "CharLSTMGenerator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_input (InputLayer)         │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ char_embedding (Embedding)      │ (None, 40, 64)         │         3,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_layer (LSTM)               │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_dropout (Dropout)          │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vocab_dense (Dense)             │ (None, 61)             │         7,869 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 331,769 (1.27 MB)

 Trainable params: 110,589 (431.99 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 221,180 (863.99 KB)

## 5. Model Training and Convergence

If training from scratch or validating the training trajectory, we train using `sparse_categorical_crossentropy`.


In [6]:
# 5.1 Training Loop (Quick Run or Full Trajectory)
# Check if model has already been trained and saved
if not os.path.exists(model_file):
    print("Training model for 10 epochs...")
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="loss", patience=2, restore_best_weights=True),
    ]
    history = model.fit(
        X,
        y,
        batch_size=128,
        epochs=10,
        callbacks=callbacks,
        verbose=1,
    )
    # Plot training loss
    plt.figure(figsize=(8, 4))
    plt.plot(history.history["loss"], marker="o", color="#1f77b4", label="Training Loss")
    plt.title("Character LSTM Training Loss", fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("Sparse Cross-Entropy Loss")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.show()
else:
    print("Trained model is already available in models/text_generator.keras!")


Trained model is already available in models/text_generator.keras!


## 6. Temperature Sampling Experiments

We test the exact same prompt at three different temperatures:
- **Low Temperature ($T = 0.4$):** Conservative, repeats dominant character n-grams.
- **Moderate Temperature ($T = 0.8$):** Realistic balance of grammatical cohesion and creativity.
- **High Temperature ($T = 1.2$):** Highly diverse, erratic words, elevated entropy.


In [7]:
# 6.1 Run Temperature Sampling Comparison
seed_prompt = "To be, or not to be, that is the question:"
temperatures = [0.4, 0.8, 1.2]
GEN_LENGTH = 200

print(f"Seed Prompt: '{seed_prompt}'\n")
print("=" * 80)

for temp in temperatures:
    generated_text = generate_text(
        model=model,
        char_to_idx=char_to_idx,
        idx_to_char=idx_to_char,
        seq_length=SEQ_LENGTH,
        seed_text=seed_prompt,
        gen_length=GEN_LENGTH,
        temperature=temp,
    )
    print(f"\n--- TEMPERATURE: {temp} ---")
    print(seed_prompt + generated_text)
    print("-" * 80)


Seed Prompt: 'To be, or not to be, that is the question:'




--- TEMPERATURE: 0.4 ---
To be, or not to be, that is the question:
When we shall have so we shall the were the people,
And the people's a consul! Cuntinges and
To his commandst the forment, to make the common
When we for the hall be so marcius, and the gates
And hea
--------------------------------------------------------------------------------



--- TEMPERATURE: 0.8 ---
To be, or not to be, that is the question:
And though we have their some a grover your very,
and the wads and for cunsues so acvesitidition;
And time not him, me ouffeed less'd conce commend
The senate, whe rivish'd son him, the awe heart
The
--------------------------------------------------------------------------------



--- TEMPERATURE: 1.2 ---
To be, or not to be, that is the question:
No me as a cause Rome?

Secat's me.
Now, Marcius, hath us.
'Te well you; that were ove
Than lables caunst betos all him,
With by have in the beave prodaringfy,
Let tervose bearf. Cound becers eised c
--------------------------------------------------------------------------------


## 7. Analysis and Observations

### Observations on Output Quality:
1. **At $T = 0.4$:** The model correctly reproduces common Shakespearean dialogue structures (e.g., character speech prefixes like `CORIOLANUS:`, `MENENIUS:`), but easily falls into repetitive word loops (*"the state of the state of the state"*).
2. **At $T = 0.8$:** Generates varied archaic English vocabulary (*"thou", "thee", "hath", "speak"*) with credible verse formatting, rhythm, and stanza line breaks.
3. **At $T = 1.2$:** Punctuation is scattered irregularly, and unusual character combinations appear because the flattened softmax distribution frequently samples low-probability tokens.

### Limitations of Character-Level Models:
- **Long-term semantic coherence:** The model maintains local syntactic validity for 5–15 characters, but cannot plan overarching plot or paragraph-level semantic intent.
- **Context window:** The 40-character window means the model has limited memory of what characters or events were mentioned earlier in the scene.
